# 18 - Silver ground truth for rating the auto-labeling service

Labels the rating frames (every 10th frame within 8 m on the 24 held-out test
traces - the same 883 frames as the human review shards) with two models that
are independent of the service under test:

1. **SAM3**, text-prompted with the video's known category, every frame on its
   own (never seeded by our detector or tracker);
2. **Qwen2.5-VL-7B**, asked to locate the named object or say it is not visible.

Agreement (boxes IoU >= 0.5, or both "not visible") becomes silver ground
truth; anything else is **disputed** and left out of the KPIs. Output:
`<artifacts>/gold/silver_v1/silver.csv`, in the format
`scripts/rate_autolabel.py` reads. A ~100-frame human audit of the silver
labels (built locally afterwards) measures how far to trust it.

Inputs: the same bundle zip as notebook 17. **GPU runtime** (L4 or A100) and
Hugging Face access to `facebook/sam3`. Both model stages save after every 50
frames and resume.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    REPO_URL = "https://github.com/Ali-RT/vision-uss-research"
    if not (REPO_DIR / ".git").exists():   # fresh VM: clone (no separate bootstrap script)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    # always run the latest main; discards VM-local edits to tracked repo files
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", "origin/main"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import shutil
import zipfile

import cv2
import numpy as np
import pandas as pd
import torch
import yaml

from vision_uss_research.labeling.corridor import masks_to_bbox, pick_target_instance
from vision_uss_research.silver_gt import (SAM3_PROMPTS, combine, parse_qwen,
                                           qwen_prompt, select_test_items)

cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / "cuv_finetune_v1.yaml").read_text())
STEP = 10                     # must match scripts/build_gold_review.py --step
AGREE_IOU = 0.5
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
SAM3_SCORE = 0.3
SMOKE = False                 # True: first 20 frames only
OUT_DIR = ARTIFACTS / "gold" / ("silver_v1_smoke" if SMOKE else "silver_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/cuv") if IN_COLAB else ARTIFACTS / "cuv_work"
WORK.mkdir(parents=True, exist_ok=True)

BUNDLE_ZIP = ARTIFACTS / cfg["bundle"]
bundle_dir = WORK / Path(cfg["bundle"]).stem
if not IN_COLAB and BUNDLE_ZIP.with_suffix("").exists():
    bundle_dir = BUNDLE_ZIP.with_suffix("")
if not (bundle_dir / "frames.csv").exists():
    assert BUNDLE_ZIP.exists(), f"upload {BUNDLE_ZIP.name} to {BUNDLE_ZIP.parent}"
    local_zip = WORK / BUNDLE_ZIP.name
    if not local_zip.exists():
        shutil.copyfile(BUNDLE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)

items = select_test_items(pd.read_csv(bundle_dir / "frames.csv"), cfg["split"], STEP)
if SMOKE:
    items = items.head(20)
items["path"] = [str(bundle_dir / "frames" / f"{s}_{t}" / f"{int(f):05d}.jpg")
                 for s, t, f in zip(items["side"], items["trace"], items["frame_idx"])]
print(f"{len(items)} frames from {items.groupby(['side', 'trace']).ngroups} test traces -> {OUT_DIR}")
display(items.groupby("label_class").size().rename("frames").to_frame())


def run_stage(name, fn, every=50):
    """Apply fn(row) -> dict to every item not yet in <OUT_DIR>/<name>.csv."""
    path = OUT_DIR / f"{name}.csv"
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["item_id"])
    todo = items[~items["item_id"].isin(set(done["item_id"]))]
    buf = []
    for k, r in enumerate(todo.itertuples(), 1):
        buf.append({"item_id": r.item_id, **fn(r)})
        if len(buf) >= every or k == len(todo):
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
            done.to_csv(path, index=False)
            buf = []
            print(f"{name}: {len(done)}/{len(items)}", flush=True)
    return done

## 1. SAM3, text prompt per frame

Prompt = the category's appearance (`SAM3_PROMPTS`); among the instances SAM3
returns, the one ahead of the bumper (approach-corridor prior, as in
notebook 02) is taken. No instance -> SAM3 says "not visible".

In [ ]:
from vision_uss_research.labeling.sam3 import Sam3Engine

sam3_path = OUT_DIR / "sam3.csv"
if not (sam3_path.exists() and len(pd.read_csv(sam3_path)) >= len(items)):
    engine = Sam3Engine(score_threshold=SAM3_SCORE)

    def sam3_one(r):
        img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
        res = engine.segment_text(img, SAM3_PROMPTS[r.label_class])
        best, diag = pick_target_instance(res, img.shape[:2])
        if best is None:
            return {"sam_x0": np.nan, "sam_y0": np.nan, "sam_x1": np.nan, "sam_y1": np.nan,
                    "sam_score": np.nan, "sam_n": 0}
        b = masks_to_bbox(res["masks"][best]) or tuple(res["boxes"][best])
        return {"sam_x0": b[0], "sam_y0": b[1], "sam_x1": b[2], "sam_y1": b[3],
                "sam_score": round(float(res["scores"][best]), 3), "sam_n": diag["n_candidates"]}

    sam3 = run_stage("sam3", sam3_one)
    del engine
    torch.cuda.empty_cache()
sam3 = pd.read_csv(sam3_path)
print(f"SAM3 found the object in {sam3['sam_x0'].notna().mean():.0%} of frames")

## 2. Qwen2.5-VL, locate the named object

Qwen answers in the pixel space of the image it was fed (resized to multiples
of 28); boxes are scaled back to 640x480. Unreadable answers are kept as such
and end up disputed.

In [ ]:
qwen_path = OUT_DIR / "qwen.csv"
if not (qwen_path.exists() and len(pd.read_csv(qwen_path)) >= len(items)):
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-vl-utils"], check=True)
    from qwen_vl_utils import process_vision_info
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL, torch_dtype=dtype, device_map="auto")
    qproc = AutoProcessor.from_pretrained(QWEN_MODEL)

    def qwen_one(r):
        msgs = [{"role": "user", "content": [{"type": "image", "image": r.path},
                                             {"type": "text", "text": qwen_prompt(r.label_class)}]}]
        text = qproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(msgs)
        inputs = qproc(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(qmodel.device)
        _, gh, gw = inputs["image_grid_thw"][0].tolist()
        with torch.inference_mode():
            out = qmodel.generate(**inputs, max_new_tokens=80, do_sample=False)
        reply = qproc.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)[0]
        vis, box = parse_qwen(reply, (gw * 14, gh * 14), (640, 480))
        row = {"qwen_visible": vis, "qwen_reply": reply.strip()[:300]}
        row.update({f"qwen_{c}": (np.nan if box is None else float(v))
                    for c, v in zip(["x0", "y0", "x1", "y1"], box if box is not None else [0] * 4)})
        return row

    qwen = run_stage("qwen", qwen_one)
    del qmodel
    torch.cuda.empty_cache()
qwen = pd.read_csv(qwen_path)
print(f"Qwen: visible {qwen['qwen_visible'].eq(True).mean():.0%}, "
      f"not visible {qwen['qwen_visible'].eq(False).mean():.0%}, unreadable {qwen['qwen_visible'].isna().mean():.0%}")

## 3. Agreement -> silver ground truth

In [ ]:
m = items.merge(sam3, on="item_id").merge(qwen, on="item_id")
rows = []
for r in m.itertuples():
    sam_box = None if pd.isna(r.sam_x0) else [r.sam_x0, r.sam_y0, r.sam_x1, r.sam_y1]
    qbox = None if pd.isna(r.qwen_x0) else np.array([r.qwen_x0, r.qwen_y0, r.qwen_x1, r.qwen_y1])
    qvis = None if pd.isna(r.qwen_visible) else bool(r.qwen_visible) if not isinstance(r.qwen_visible, str) \
        else r.qwen_visible.lower() == "true"
    c = combine(sam_box, qvis, qbox, (640, 480), AGREE_IOU)
    rows.append({"item_id": r.item_id, "side": r.side, "trace": r.trace, "frame_idx": r.frame_idx,
                 "category": r.label_class, "reviewer": "silver_sam3_qwen", **c,
                 "dist_m": r.dist_m, "time": ""})
silver = pd.DataFrame(rows)
silver.to_csv(OUT_DIR / "silver_detail.csv", index=False)
cols = ["item_id", "side", "trace", "frame_idx", "category", "reviewer", "status",
        "x0", "y0", "x1", "y1", "truncated", "time"]
silver[cols].to_csv(OUT_DIR / "silver.csv", index=False)

summary = silver.pivot_table(index="category", columns="status", values="item_id",
                             aggfunc="count", fill_value=0)
summary["silver share"] = (1 - summary.get("disputed", 0) / summary.sum(axis=1)).round(2)
display(summary)
display(silver[silver["status"] == "disputed"].groupby(["category", "reason"]).size()
        .rename("disputed frames").to_frame())
print(f"silver ground truth: {(silver['status'] != 'disputed').sum()}/{len(silver)} frames -> {OUT_DIR / 'silver.csv'}")

In [ ]:
# look before trusting: agreed boxes (green) and disputed frames (SAM3 red, Qwen blue)
import matplotlib.pyplot as plt

def gallery(df, title, n=8):
    if df.empty:
        print(f"{title}: none"); return
    pick = df.sample(min(n, len(df)), random_state=0).merge(m[["item_id", "path", "sam_x0", "sam_y0", "sam_x1", "sam_y1",
                                                               "qwen_x0", "qwen_y0", "qwen_x1", "qwen_y1"]], on="item_id")
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, pick.itertuples()):
        img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
        if r.status == "box":
            cv2.rectangle(img, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (0, 200, 0), 3)
        else:
            if not pd.isna(r.sam_x0):
                cv2.rectangle(img, (int(r.sam_x0), int(r.sam_y0)), (int(r.sam_x1), int(r.sam_y1)), (230, 40, 40), 3)
            if not pd.isna(r.qwen_x0):
                cv2.rectangle(img, (int(r.qwen_x0), int(r.qwen_y0)), (int(r.qwen_x1), int(r.qwen_y1)), (40, 90, 230), 3)
        ax.imshow(img)
        ax.set_title(f"{r.category} {r.dist_m:.1f} m {r.status} {r.reason}", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

gallery(silver[silver["status"] == "box"], "silver boxes")
gallery(silver[silver["status"] == "disputed"], "disputed")

## Next

1. Download `silver.csv` (and `silver_detail.csv`) from `<artifacts>/gold/silver_v1/`.
2. Locally, build the audit page (100 random silver frames, the silver box shown;
   the reviewer accepts with `A` or redraws):
   `uv run python scripts/build_gold_review.py --prefill silver.csv --sample 100 --name audit_v1 --reviewers 1 --overlap 0`
3. Rate: `uv run --with tabulate python scripts/rate_autolabel.py --gold-dir <folder with silver.csv> --audit-dir <folder with the audit CSV>`.
   The report gives every KPI against silver plus the audited silver accuracy.